In [2]:
import pandas as pd

df = pd.read_csv("http://114.207.245.181:13000/csv/supply.csv")
df

,수요량,생산능력,재고량,월,공급량
0,950,1000,200,1월,250
1,1050,1100,180,2월,230
2,980,1000,150,3월,170
3,1020,1050,130,4월,160
4,1100,1150,140,5월,190
5,1200,1250,160,6월,210
6,1400,1450,170,7월,220
7,1350,1400,160,8월,210
8,1300,1350,150,9월,200
9,1250,1300,140,10월,190


In [3]:
df.columns

Index(['수요량', '생산능력', '재고량', '월', '공급량'], dtype='object')

In [6]:
x = df[['수요량', '생산능력', '재고량']].values
y = df[['공급량']].values

x.shape, y.shape

((12, 3), (12, 1))

In [ ]:
# 0~1 사이의 숫자로 정규화(스케일)
from sklearn.preprocessing import MinMaxScaler
x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

x_scaled = x_scaler.fit_transform(x)
y_scaled = y_scaler.fit_transform(y)

x_scaled[:3], y_scaled[:3]

(array([[0.        , 0.        , 1.        ],
        [0.22222222, 0.22222222, 0.75      ],
        [0.06666667, 0.        , 0.375     ]]),
 array([[1.        ],
        [0.77777778],
        [0.11111111]]))

In [ ]:
import numpy as np
timesteps = 3   # 3개월 단위로
x = []
y = []

for i in range(len(df) - timesteps):
    x.append(x_scaled[i:i+timesteps])
    y.append(y_scaled[i+timesteps])

x = np.array(x)
y = np.array(y)
x.shape, y.shape

((9, 3, 3), (9, 1))

In [ ]:
# 훈련 세트와 테스트 세트 나누기
split = int(len(x) * 0.8)

x_train = x[:split]
x_test = x[split:]

y_train = y[:split]
y_test = y[split:]

x_train.shape, x_test.shape, y_train.shape, y_test.shape

((7, 3, 3), (2, 3, 3), (7, 1), (2, 1))

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import LSTM, Dense, Input
from tensorflow.keras.models import Sequential

model = Sequential([
    Input(shape=(3,3)),
    # (3*10) + (10*10) + 10 => 140*4gate
    LSTM(units=10, activation="tanh", return_sequences=True),
    # (10*50) + (50*50) + 50 => 3050*4
    LSTM(units=50, activation="relu", return_sequences=False),
    Dense(units=1, activation="linear")
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                   │ (None, 3, 10)          │           560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 50)             │        12,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,811 (50.04 KB)

 Trainable params: 12,811 (50.04 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

In [18]:
model.fit(x_train, y_train, epochs=2000, validation_data=(x_test, y_test))

Epoch 1/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - loss: 0.2089 - mae: 0.4108 - val_loss: 0.2422 - val_mae: 0.4389
Epoch 2/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - loss: 0.2058 - mae: 0.4082 - val_loss: 0.2387 - val_mae: 0.4348
Epoch 3/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 0.2027 - mae: 0.4055 - val_loss: 0.2352 - val_mae: 0.4306
Epoch 4/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - loss: 0.1996 - mae: 0.4027 - val_loss: 0.2316 - val_mae: 0.4263
Epoch 5/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - loss: 0.1965 - mae: 0.3999 - val_loss: 0.2280 - val_mae: 0.4220
Epoch 6/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - loss: 0.1934 - mae: 0.3971 - val_loss: 0.2244 - val_mae: 0.4176
Epoch 7/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - loss: 0.1904 - mae: 0.3942 - val_loss: 0.2208 - val_mae: 0.4132
Epoch 8/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - loss: 0.1873 - mae: 0.3913 - val_loss: 0.2172 - val_mae: 0.4086
Epoch 9/2000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - loss: 0.1842 

In [19]:
from sklearn.metrics import r2_score
y_pred_scaled = model.predict(x_test)

y_pred = y_scaler.inverse_transform(y_pred_scaled)
y_real = y_scaler.inverse_transform(y_test)

r2_score(y_real, y_pred)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step


0.30883040496468306

In [20]:
last_3_month = x_scaled[-3:]
sample = last_3_month.reshape(-1, 3, 3)
pred_sample = model.predict(sample)
y_scaler.inverse_transform(pred_sample)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step


array([[217.16304]], dtype=float32)